# 8 — One value, N parameters

Chapter 3 proved the thesis at the cache: change a captured *value* and
`compiles == 1`. This chapter builds the machinery that makes it true at the
**byte** level. M0's disease was the per-frame flatten — every call re-walked
the environment hunting for its uniforms. The cure is structural, and it is
**bidirectional** (design 040 §3b — kernels are destination-passing at the
ABI):

- a **PackPlan** is built once per cache entry, from **types alone** —
  values never shape a plan;
- per call, one generic loop writes leaf values into a reused **staging**
  buffer (byte slots) or out the **leaves channel** (buffer-class leaves,
  when ndarrays arrive);
- a **ResultPlan** mirrors it outbound: destinations allocated from the
  result type, result bytes unflattened back into the logical value;
- and the decision is **printable IR**: logical `core.env` ops legalize into
  physical `abi.slot` ops, so marshaling is golden-testable, never folklore.

Source: `src/pdum/dsl/kernel/pack.py`. New words for the glossary: *leaf*,
*slot*, *plan*, *staging*, *leaves channel*.

In [1]:
import ast
import struct

from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.api import jit
from pdum.dsl.kernel.lower import lower_handle
from pdum.dsl.kernel.ops import CORE_OPS
from pdum.dsl.kernel.pack import (
    ABI_OPS,
    NORMALIZE_ENV,
    build_extractor,
    legalize_params,
    pack_into,
    plan_from_types,
    result_plan,
    unpack_result,
)
from pdum.dsl.kernel.printer import print_program
from pdum.dsl.kernel.rewrite import MatchLog, run_stage
from pdum.dsl.kernel.valuekind import BUILTINS
from pdum.dsl.stdlib.base_lang import LOWER_RULES

ALL_OPS = {**CORE_OPS, **ABI_OPS}


def _subscript(ctx, node):  # stand-in: const-index tuple access (tuples land at step 10)
    return ctx.emit("core.extract", ctx.lower(node.value), node=node, index=node.slice.value)


RULES = dict(LOWER_RULES)
RULES[ast.Subscript] = _subscript


def make_shader(cx, cy, gain):
    center = (cx, cy)

    @jit()
    def bright(v):
        return v * gain

    @jit()
    def shader(x):
        return bright(x - center[0]) + center[1]

    return shader


sh = make_shader(0.5, 0.25, 2.0)
region = lower_handle(sh, RULES, ALL_OPS, arg_types=(T.f64,))
print(print_program(region, name="shader"))

shader(%p0: f64) {
  %0 = core.env {slot = (1,)} : (f64, f64)
  %1 = core.extract %0 {index = 0} : f64
  %2 = core.sub %p0, %1 : f64
  %3 = core.env {slot = (0, 0)} : f64
  %4 = core.mul %2, %3 : f64
  %5 = core.env {slot = (1,)} : (f64, f64)
  %6 = core.extract %5 {index = 1} : f64
  %7 = core.add %4, %6 : f64
  core.yield %7
}


A capture worth marshaling: a **tuple** (`center`) and a **nested kernel**
(`bright`, which itself captures `gain`). Note the `core.extract` ops eating
the tuple env, and the `(0, 0)`-style env paths from ch07's inlining.

## The leaf walk — from types alone

Before any value is touched, the *types* already determine every physical
parameter. `leaf_entries` is the static walk; the `FnType` walker is the
architecture's **EnvLeaf recursion** — a captured kernel's leaves are its
env's leaves, paths prefixed. These are the same paths lowering stamps on
`core.env`.

In [2]:
for sub, leaf in BUILTINS.leaf_entries(sh.fntype):
    print(f"  env path {str(sub):<8} {leaf}")
print()
print("(0, 0) is `gain` — reached THROUGH the captured `bright` kernel.")

  env path (0, 0)   ScalarLeaf(kind='f64')
  env path (1, 0)   ScalarLeaf(kind='f64')
  env path (1, 1)   ScalarLeaf(kind='f64')

(0, 0) is `gain` — reached THROUGH the captured `bright` kernel.


## Normalizing: composite captures become leaf paths

`core.extract`-of-an-env folds into the env's *path* — after this stage,
every surviving capture is leaf-typed. This is a rewrite Stage like any
other (ch06's driver, ch06's MatchLog); marshaling adds no new mechanism.

In [3]:
log = MatchLog()
normed = run_stage(region, NORMALIZE_ENV, ALL_OPS, log=log)
for stage, old, new in log.entries:
    print(f"  [{stage}] {old.op} {dict(old.attrs)}  ->  {new.op} {dict(new.attrs)}")
print()
print(print_program(normed, name="shader"))

  [normalize-env] core.extract {'index': 0}  ->  core.env {'slot': (1, 0)}
  [normalize-env] core.extract {'index': 1}  ->  core.env {'slot': (1, 1)}

shader(%p0: f64) {
  %0 = core.env {slot = (1, 0)} : f64
  %1 = core.sub %p0, %0 : f64
  %2 = core.env {slot = (0, 0)} : f64
  %3 = core.mul %1, %2 : f64
  %4 = core.env {slot = (1, 1)} : f64
  %5 = core.add %3, %4 : f64
  core.yield %5
}


## The plan: types in, offsets out

`plan_from_types` walks env types then arg types and asks the layout policy
(`packed_dest`, the reference dense layout — real backends bring std140 and
friends at step 9) for a destination per leaf. **No value was consulted.**
The argument travels the same road as the captures — it is just the `arg`
root.

In [4]:
plan = plan_from_types(sh.env_types, (T.f64,), BUILTINS)
print(f"  {'source':<18} {'offset':>6}  fmt")
for spec in plan.slots:
    src = f"{spec.source.root}[{spec.source.index}]{list(spec.source.sub)}"
    print(f"  {src:<18} {spec.dest.offset:>6}  {spec.dest.fmt}")
print(f"  staging = {plan.staging_size} bytes; SlotSpec.convert (all None today) is the units seat")

  source             offset  fmt
  env[0][0]               0  <d
  env[1][0]               8  <d
  env[1][1]              16  <d
  arg[0][]               24  <d
  staging = 32 bytes; SlotSpec.convert (all None today) is the units seat


## Legalizing: the marshaling decision becomes IR

`legalize_params(plan)` rewrites every logical `core.env` into a physical
`abi.slot` carrying its byte offset and format.

The stage then proves its own point. A conversion target (`legal={core, abi}`
— which *dialects* may remain) **cannot** express "no capture survived":
`core.env` **is** `core`, so it would sail straight through. That is why the
stage also declares `forbid={"core.env"}` — an op-level check, run after the
rules reach fixpoint. Per-frame flattening is structurally impossible because
a machine checks it, not because the rule set happens to be total today. (The
first draft of this chapter claimed the namespace set did the work; the step-7
review caught the lie, and `Stage.forbid` is the fix.)

In [5]:
final = run_stage(normed, legalize_params(plan), ALL_OPS)
print(print_program(final, name="shader"))
print()
print("artifact key:", final.key.hex()[:16], "… (offsets are in it: layout is identity)")

shader(%p0: f64) {
  %0 = abi.slot {fmt = '<d', offset = 8, src = ('env', 1, 0)} : f64
  %1 = core.sub %p0, %0 : f64
  %2 = abi.slot {fmt = '<d', offset = 0, src = ('env', 0, 0)} : f64
  %3 = core.mul %1, %2 : f64
  %4 = abi.slot {fmt = '<d', offset = 16, src = ('env', 1, 1)} : f64
  %5 = core.add %3, %4 : f64
  core.yield %5
}

artifact key: 6f9c04d11e658a10 … (offsets are in it: layout is identity)


## The payoff, in hex

Build the extractor once, then move only bytes. `build_extractor` compiles
the plan's leaf paths into **one getter per slot** — a chain of pure
index/attribute reads resolved against the *types* at build time — so the
per-call path does no kind dispatch and no recursive walk. (That is the
other half of killing M0's per-frame flatten: the IR no longer hunts for
uniforms, and neither does the extractor.) A second shader instance with
**different values but the same types** reuses the plan, the extractor, and
the artifact key untouched — the difference between the two calls is exactly
the staging bytes.

In [6]:
def hexdump(buf):
    return " ".join(f"{b:02x}" for b in buf)


extract = build_extractor(sh.env_types, (T.f64,), plan, BUILTINS)
staging = bytearray(plan.staging_size)
pack_into(plan, staging, extract(sh.captures, (0.0,)))
print("gain=2.0, center=(0.5, 0.25):")
print("  ", hexdump(staging))

sh2 = make_shader(0.5, 0.25, 9.0)  # new VALUES, same types
final2 = run_stage(
    run_stage(lower_handle(sh2, RULES, ALL_OPS, arg_types=(T.f64,)), NORMALIZE_ENV, ALL_OPS),
    legalize_params(plan),
    ALL_OPS,
)
staging2 = bytearray(plan.staging_size)
pack_into(plan, staging2, extract(sh2.captures, (0.0,)))  # SAME compiled extractor
print("gain=9.0, same types:")
print("  ", hexdump(staging2))
print()
print("plan reused:      ", plan == plan_from_types(sh2.env_types, (T.f64,), BUILTINS))
print("same artifact key:", final.key == final2.key)
print("same bytes:       ", bytes(staging) == bytes(staging2))

gain=2.0, center=(0.5, 0.25):
   00 00 00 00 00 00 00 40 00 00 00 00 00 00 e0 3f 00 00 00 00 00 00 d0 3f 00 00 00 00 00 00 00 00
gain=9.0, same types:
   00 00 00 00 00 00 22 40 00 00 00 00 00 00 e0 3f 00 00 00 00 00 00 d0 3f 00 00 00 00 00 00 00 00

plan reused:       True
same artifact key: True
same bytes:        False


## The output mirror: results are destinations

Kernels don't *return* values at the ABI — they write into destinations
they were handed (a compute kernel's out-buffer, a fragment's render
target). The functional `y = f(x)` stays the language-level truth; the
bridge is the **ResultPlan**: destinations allocated from the result type,
device bytes unflattened back into the logical value. Bidirectional from
the start — the output half is not bolted on later (040 §3b). Destinations
get reused across calls while types hold, same as staging.

In [7]:
rp = result_plan(T.Tuple((T.f64, T.f64)), BUILTINS)
out = bytearray(rp.size)
struct.pack_into("<d", out, 0, 0.125)  # stand-in for the device writing its destinations
struct.pack_into("<d", out, 8, 0.875)
print("destinations:", [(s.dest.offset, s.dest.fmt) for s in rp.slots], "| size", rp.size)
print("unflattened :", unpack_result(rp, out, BUILTINS))

destinations: [(0, '<d'), (8, '<d')] | size 16
unflattened : (0.125, 0.875)


In [8]:
from pdum.dsl import viz

viz.install()
final  # hover a node: type + provenance survived all the way to the ABI stage

Region(params=(Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 0),), loc=None),), body=(Node(op='core.yield', type=f64, args=(Node(op='core.add', type=f64, args=(Node(op='core.mul', type=f64, args=(Node(op='core.sub', type=f64, args=(Node(op='core.param', type=f64, args=(), regions=(), attrs=(('index', 0),), loc=None), Node(op='abi.slot', type=f64, args=(), regions=(), attrs=(('fmt', '<d'), ('offset', 8), ('src', ('env', 1, 0))), loc=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83217/4101649394.py', line=43, col=22))), regions=(), attrs=(), loc=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83217/4101649394.py', line=43, col=18)), Node(op='abi.slot', type=f64, args=(), regions=(), attrs=(('fmt', '<d'), ('offset', 0), ('src', ('env', 0, 0))), loc=CallLoc(callee=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83217/4101649394.py', line=39, col=15), caller=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83217/4101649394.py', line=43, col=11)))), regions=(), attrs=(), loc=CallLoc(callee=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83217/4101649394.py', line=39, col=11), caller=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83217/4101649394.py', line=43, col=11))), Node(op='abi.slot', type=f64, args=(), regions=(), attrs=(('fmt', '<d'), ('offset', 16), ('src', ('env', 1, 1))), loc=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83217/4101649394.py', line=43, col=35))), regions=(), attrs=(), loc=Loc(file='/var/folders/vw/tdwty7n902d81s2lj6nz2y9w0000gn/T/ipykernel_83217/4101649394.py', line=43, col=11)),), regions=(), attrs=(), loc=None),))

## Things to notice

- Every `abi.slot`'s `src` matches an env path lowering stamped in ch07 —
  three layers (capture, lowering, marshaling) agree on one path language.
- The artifact key *changed* at legalization (offsets entered the IR):
  layout is identity at the artifact tier, exactly the two-tier law.
- `build_extractor` never calls `flatten` on the hot path — it compiles the
  plan's paths into getters. `flatten` remains the *reference* semantics, and
  a fuzz asserts the two roads agree; a kind whose `child` and `leaves`
  aspects disagreed would write the wrong value into the right slot, which no
  count check could ever see.
- `SlotSpec.convert` is `None` in every slot today. Step 15 puts an
  `Affine` mm→inch converter there: bytes change per call, plan and
  artifact never do — a unit tweak that cannot recompile.

## What we can't do yet

No `FastRecord` is wired: nothing *runs* the plan on a cache hit until the
Python backend and the hot path land (ch09 — `Registry` v1, guards armed,
`Handle.__call__`, the thesis test under `no_compile`). `core.param`s keep
their logical spelling: the physical calling convention is the renderer's
half of the deal (steps 8/9). And the leaves channel is empty until
ndarrays bring `BufferLeaf` (ch12).